# KOSPI200 지수옵션 체결 데이터 전용 수집기
키움 Open API를 사용하여 KOSPI200 지수옵션 **체결 데이터**(호가 제외) 및 KOSPI200 지수 데이터를 CSV로 저장

### 기존 호가+체결 수집기와의 차이점
- 호가 데이터 제외 (매수/매도호가, 잔량 등)
- 체결 데이터 + 그릭스 + 지수 데이터만 수집
- CSV 컬럼 수: 38개 → 15개 (약 60% 감소)
- 체결방향 계산 불가 (호가 비교 필요)

## 1. 설정

In [ ]:
# === 사용자 설정 ===
OPTION_CODE = "B0166830"  # KOSPI200 지수옵션 코드
INDEX_CODE = "201"        # KOSPI200 업종지수 (업종코드)
SCREEN_NO = "1000"        # 화면번호

## 2. 라이브러리 Import

In [2]:
import sys
import os
from datetime import datetime
import pandas as pd
from PyQt5.QtWidgets import QApplication
from PyQt5.QAxContainer import QAxWidget
from PyQt5.QtCore import QEventLoop, QTimer

## 3. CSV 저장 설정 및 FID 매핑 (체결 전용)

In [3]:
# 저장 디렉토리 설정
DATA_DIR = os.path.join(os.getcwd(), "data")
os.makedirs(DATA_DIR, exist_ok=True)

# CSV 파일 경로 생성
timestamp_str = datetime.now().strftime("%Y%m%d_%H%M")
CSV_FILENAME = f"execution_{OPTION_CODE}_{timestamp_str}.csv"
CSV_PATH = os.path.join(DATA_DIR, CSV_FILENAME)

# CSV 컬럼 정의 (체결 전용 - 15개)
CSV_COLUMNS = [
    # 기본 정보 (2개)
    "시간", "데이터유형",
    
    # 옵션 체결 (5개)
    "옵션코드", "옵션체결시간", "옵션현재가",
    "옵션누적거래량", "옵션체결수량",
    
    # 옵션 그릭스 (5개)
    "옵션이론가", "옵션IV", "옵션델타", "옵션감마", "옵션미결제약정",
    
    # 지수 (3개)
    "지수코드", "지수현재가", "지수등락률",
    
    # 파생 컬럼 (1개)
    "1초체결량",
]

# FID 매핑 - 옵션 체결 데이터
FID_OPTION_TRADE = {
    "옵션체결시간": 20,
    "옵션현재가": 10,
    "옵션누적거래량": 13,
    "옵션체결수량": 15,
}

# FID 매핑 - 옵션 그릭스
FID_OPTION_GREEKS = {
    "옵션이론가": 182,
    "옵션IV": 190,
    "옵션델타": 191,
    "옵션감마": 192,
    "옵션미결제약정": 195,
}

# FID 매핑 - 업종지수 (KOSPI200 지수)
FID_INDEX = {
    "지수현재가": 10,
    "지수등락률": 12,
}

# 전체 FID 통합 (실시간 등록용) - 호가 FID 제외
ALL_OPTION_FIDS = set(FID_OPTION_TRADE.values()) | set(FID_OPTION_GREEKS.values())
ALL_INDEX_FIDS = set(FID_INDEX.values())

print(f"체결 전용 수집기")
print(f"  - CSV 컬럼 수: {len(CSV_COLUMNS)}개")
print(f"  - 옵션 FID 수: {len(ALL_OPTION_FIDS)}개")
print(f"  - 지수 FID 수: {len(ALL_INDEX_FIDS)}개")

체결 전용 수집기
  - CSV 컬럼 수: 16개
  - 옵션 FID 수: 9개
  - 지수 FID 수: 2개


## 4. KiwoomTradeOnlyCollector 클래스 정의

In [4]:
class KiwoomTradeOnlyCollector:
    """KOSPI200 지수옵션 체결 데이터 전용 수집기"""
    
    def __init__(self, option_code, index_code, screen_no, csv_path, interval_ms=1000):
        self.option_code = option_code
        self.index_code = index_code
        self.screen_no = screen_no
        self.csv_path = csv_path
        self.interval_ms = interval_ms

        self.app = QApplication(sys.argv)
        self.kiwoom = QAxWidget("KHOPENAPI.KHOpenAPICtrl.1")

        self.kiwoom.OnEventConnect.connect(self.on_event_connect)
        self.kiwoom.OnReceiveRealData.connect(self.on_receive_real_data)

        self.login_event_loop = QEventLoop()
        self.connected = False
        self.data_count = 0

        # 마지막 수신 데이터 캐시 (호가 필드 제외)
        self.last_data = {col: "" for col in CSV_COLUMNS}
        self.last_data["옵션코드"] = option_code
        self.last_data["지수코드"] = index_code
        
        # 이전 스냅샷 저장용 (1초 체결량 계산용)
        self.prev_cumulative_volume = 0

        # 데이터 수신 여부 플래그
        self.data_received = False

        # 수신된 모든 real_type 기록 (디버깅용)
        self.received_types = set()

        # 1초 타이머 설정
        self.timer = QTimer()
        self.timer.timeout.connect(self.on_timer)

        # CSV 헤더 작성
        self._init_csv()

    def _init_csv(self):
        """CSV 파일 초기화 (헤더 작성)"""
        df = pd.DataFrame(columns=CSV_COLUMNS)
        df.to_csv(self.csv_path, index=False, encoding="utf-8-sig")
        print(f"CSV 파일 생성: {self.csv_path}")
    
    def _safe_int(self, value):
        """안전한 정수 변환"""
        try:
            return int(value) if value else 0
        except (ValueError, TypeError):
            return 0
    
    def _compute_derived_fields(self):
        """1초 체결량 계산 (체결방향은 호가 없이 판단 불가)"""
        current_vol = self._safe_int(self.last_data.get('옵션누적거래량'))
        volume_delta = max(0, current_vol - self.prev_cumulative_volume)
        
        self.last_data['1초체결량'] = volume_delta
        self.prev_cumulative_volume = current_vol

    def login(self):
        """로그인"""
        self.kiwoom.dynamicCall("CommConnect()")
        self.login_event_loop.exec_()

    def on_event_connect(self, err_code):
        """로그인 결과 처리"""
        if err_code == 0:
            print("\n[성공] 로그인 완료")
            self.connected = True

            user_name = self.kiwoom.dynamicCall("GetLoginInfo(QString)", "USER_NAME")
            server_type = self.kiwoom.dynamicCall("GetLoginInfo(QString)", "GetServerGubun")

            print(f"사용자: {user_name}")
            print(f"서버: {'모의투자' if server_type == '1' else '실투자'}")
        else:
            print(f"\n[실패] 로그인 실패 (에러코드: {err_code})")
            self.connected = False

        self.login_event_loop.exit()

    def set_real_reg(self):
        """실시간 등록 (체결 + 그릭스만, 호가 제외)"""
        option_fid_list = ";".join([str(fid) for fid in ALL_OPTION_FIDS])
        index_fid_list = ";".join([str(fid) for fid in ALL_INDEX_FIDS])

        # 옵션 종목 등록 (체결 + 그릭스 FID만)
        result1 = self.kiwoom.dynamicCall(
            "SetRealReg(QString, QString, QString, QString)",
            self.screen_no, self.option_code, option_fid_list, "0"
        )
        print(f"옵션 실시간 등록 ({self.option_code}): {result1}")
        print(f"  등록 FID (호가 제외): {option_fid_list}")

        # KOSPI200 지수 추가 등록
        result2 = self.kiwoom.dynamicCall(
            "SetRealReg(QString, QString, QString, QString)",
            self.screen_no, self.index_code, index_fid_list, "1"
        )
        print(f"지수 실시간 등록 ({self.index_code}): {result2}")

        return result1, result2

    def get_real_data(self, code, fid):
        """실시간 데이터 조회"""
        value = self.kiwoom.dynamicCall(
            "GetCommRealData(QString, int)", code, fid
        ).strip()
        return value

    def _clean_value(self, value):
        """값 정리 (부호 제거)"""
        if value and (value.startswith("+") or value.startswith("-")):
            if value[1:].replace(".", "").isdigit():
                return value[1:]
        return value

    def on_receive_real_data(self, code, real_type, real_data):
        """실시간 데이터 수신 → 캐시 업데이트 (호가 타입 무시)"""

        # [디버깅] 새로운 real_type 출력
        if real_type not in self.received_types:
            self.received_types.add(real_type)
            print(f"\n>>> [NEW TYPE] real_type = '{real_type}' (code: {code}) <<<")

        # 호가 타입은 무시
        if real_type in ("옵션호가잔량", "주식옵션호가잔량", "지수옵션호가잔량", 
                         "선물옵션호가잔량", "선물호가잔량"):
            return

        # 옵션이론가 처리 (그릭스)
        if real_type == "옵션이론가" and code == self.option_code:
            for key, fid in FID_OPTION_GREEKS.items():
                value = self._clean_value(self.get_real_data(code, fid))
                if value:
                    self.last_data[key] = value
            self.data_received = True

        # 옵션시세 처리 (체결)
        elif real_type in ("옵션시세", "주식옵션시세", "지수옵션시세", 
                           "선물옵션시세", "선물시세") and code == self.option_code:
            for key, fid in FID_OPTION_TRADE.items():
                value = self._clean_value(self.get_real_data(code, fid))
                if value:
                    self.last_data[key] = value
            self.data_received = True

        # 업종지수 처리 (KOSPI200)
        elif real_type in ("업종지수", "업종등락") and code == self.index_code:
            for key, fid in FID_INDEX.items():
                value = self._clean_value(self.get_real_data(code, fid))
                if value:
                    self.last_data[key] = value
            self.data_received = True

        # 옵션 코드 매칭 시 체결/그릭스 데이터 추출 시도
        if code == self.option_code:
            for key, fid in FID_OPTION_TRADE.items():
                value = self._clean_value(self.get_real_data(code, fid))
                if value:
                    self.last_data[key] = value

    def on_timer(self):
        """1초마다 현재 캐시 상태를 CSV에 저장"""
        if not self.data_received:
            return
        
        # 파생 컬럼 계산 (1초 체결량)
        self._compute_derived_fields()

        # 타임스탬프 업데이트
        self.last_data["시간"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        self.last_data["데이터유형"] = "스냅샷"

        # CSV에 append
        self.data_count += 1
        df = pd.DataFrame([self.last_data])
        df.to_csv(self.csv_path, mode="a", header=False, index=False, encoding="utf-8-sig")

        # 콘솔 출력
        vol_delta = self.last_data.get('1초체결량', 0)
        cumvol = self.last_data.get('옵션누적거래량', '')
        iv = self.last_data.get('옵션IV', '')
        index_price = self.last_data.get('지수현재가', '')
        
        print(f"[{self.data_count:4d}] {self.last_data['시간']} | "
              f"옵션: {self.last_data.get('옵션현재가', ''):>8} | "
              f"IV: {iv:>8} | "
              f"누적: {cumvol:>6} | "
              f"체결: {vol_delta:>3} | "
              f"지수: {index_price}")

    def run(self):
        """메인 실행"""
        if not self.option_code:
            print("[오류] OPTION_CODE가 설정되지 않았습니다.")
            return

        self.login()

        if not self.connected:
            print("로그인 실패로 종료합니다.")
            return

        print("\n" + "=" * 60)
        print(f"KOSPI200 지수옵션 체결 데이터 전용 수집 시작")
        print(f"  - 옵션: {self.option_code}")
        print(f"  - 지수: {self.index_code} (KOSPI200)")
        print(f"  - 저장 간격: {self.interval_ms}ms")
        print(f"  - 호가 데이터: 제외")
        print("=" * 60)

        self.set_real_reg()

        # 타이머 시작
        self.timer.start(self.interval_ms)

        print(f"\n실시간 데이터 수집 중... (중지: Kernel Interrupt)")
        print(f"저장 위치: {self.csv_path}")
        print(f"저장 방식: {self.interval_ms/1000:.1f}초마다 스냅샷 저장\n")

        self.app.exec_()

    def stop(self):
        """수집 중지"""
        self.timer.stop()
        print(f"\n수집 종료. 총 {self.data_count}개 데이터 저장됨.")
        print(f"저장 파일: {self.csv_path}")
        print(f"수신된 타입 목록: {self.received_types}")
        self.app.quit()

## 5. 실행

In [ ]:
# 수집기 생성 및 실행
collector = KiwoomTradeOnlyCollector(
    option_code=OPTION_CODE,
    index_code=INDEX_CODE,
    screen_no=SCREEN_NO,
    csv_path=CSV_PATH,
    interval_ms=1000  # 1초마다 스냅샷 저장
)

# 실행 (중지하려면 Kernel > Interrupt)
collector.run()

CSV 파일 생성: c:\Users\adg01\OneDrive\바탕 화면\Yonsei\26-1 Yonsei\Y-FoRM\Y-FoRM 장플\option,futures MM\crawling\data\execution_B0166750_20260331_1445.csv

[성공] 로그인 완료
사용자: 박상혁
서버: 실투자

KOSPI200 지수옵션 체결 데이터 전용 수집 시작
  - 옵션: B0166750
  - 지수: 201 (KOSPI200)
  - 저장 간격: 1000ms
  - 호가 데이터: 제외
옵션 실시간 등록 (B0166750): 0
  등록 FID (호가 제외): 192;195;10;13;15;20;182;190;191
지수 실시간 등록 (201): 0

실시간 데이터 수집 중... (중지: Kernel Interrupt)
저장 위치: c:\Users\adg01\OneDrive\바탕 화면\Yonsei\26-1 Yonsei\Y-FoRM\Y-FoRM 장플\option,futures MM\crawling\data\execution_B0166750_20260331_1445.csv
저장 방식: 1.0초마다 스냅샷 저장


>>> [NEW TYPE] real_type = '업종지수' (code: 201) <<<

>>> [NEW TYPE] real_type = '옵션이론가' (code: B0166750) <<<
[   1] 2026-03-31 14:45:54 | 옵션:          | IV:  55.8829 | 누적:        | 체결:   0 | 지수: 752.93
[   2] 2026-03-31 14:45:55 | 옵션:          | IV:  55.9263 | 누적:        | 체결:   0 | 지수: 753.11
[   3] 2026-03-31 14:45:56 | 옵션:          | IV:  55.8853 | 누적:        | 체결:   0 | 지수: 752.94
[   4] 2026-03-31 14:45:57 | 옵션:    

KeyboardInterrupt: 

[ 423] 2026-03-31 14:52:58 | 옵션:          | IV:  56.0711 | 누적:        | 체결:   0 | 지수: 753.71
[ 424] 2026-03-31 14:52:59 | 옵션:          | IV:  56.0566 | 누적:        | 체결:   0 | 지수: 753.65
[ 425] 2026-03-31 14:53:00 | 옵션:          | IV:  56.0687 | 누적:        | 체결:   0 | 지수: 753.70
[ 426] 2026-03-31 14:53:01 | 옵션:          | IV:  56.0735 | 누적:        | 체결:   0 | 지수: 753.72
[ 427] 2026-03-31 14:53:02 | 옵션:          | IV:  56.0494 | 누적:        | 체결:   0 | 지수: 753.62
[ 428] 2026-03-31 14:53:03 | 옵션:          | IV:  56.0856 | 누적:        | 체결:   0 | 지수: 753.77
[ 429] 2026-03-31 14:53:04 | 옵션:          | IV:  56.0277 | 누적:        | 체결:   0 | 지수: 753.53
[ 430] 2026-03-31 14:53:05 | 옵션:          | IV:  56.1073 | 누적:        | 체결:   0 | 지수: 753.86
[ 431] 2026-03-31 14:53:06 | 옵션:          | IV:  56.1000 | 누적:        | 체결:   0 | 지수: 753.83
[ 432] 2026-03-31 14:53:07 | 옵션:          | IV:  56.0614 | 누적:        | 체결:   0 | 지수: 753.67


## 6. 수집된 데이터 확인

In [ ]:
import pandas as pd

# 데이터 로드
try:
    df = pd.read_csv(CSV_PATH, encoding="utf-8-sig")
    print(f"파일: {CSV_PATH}")
except:
    # 최근 파일 찾기
    import glob
    files = glob.glob("data/execution_*.csv")
    if files:
        latest_file = max(files, key=os.path.getmtime)
        df = pd.read_csv(latest_file, encoding="utf-8-sig")
        print(f"파일: {latest_file}")
    else:
        print("데이터 파일을 찾을 수 없습니다.")
        df = None

if df is not None and len(df) > 0:
    print(f"총 행 수: {len(df)}")
    print(f"수집 시간 범위: {df['시간'].iloc[0]} ~ {df['시간'].iloc[-1]}")
    print(f"\n=== 컬럼별 데이터 유무 ===")
    for col in df.columns:
        non_null = df[col].notna().sum()
        non_empty = (df[col] != "").sum() if df[col].dtype == object else non_null
        print(f"{col}: {non_empty}/{len(df)} ({non_empty/len(df)*100:.1f}%)")
else:
    print("데이터가 없습니다.")

In [ ]:
# 체결 데이터 확인
if df is not None and len(df) > 0:
    print("=== 체결 데이터 (최근 10건) ===")
    display(df[["시간", "옵션현재가", "옵션누적거래량", "1초체결량", 
                "옵션IV", "옵션델타", "지수현재가"]].tail(10))

In [ ]:
# 체결량 분석
if df is not None and len(df) > 0:
    print("=== 체결량 통계 ===")
    trade_vol = df['1초체결량'].astype(int)
    print(f"총 체결량: {trade_vol.sum():,}")
    print(f"평균 체결량/초: {trade_vol.mean():.2f}")
    print(f"최대 체결량/초: {trade_vol.max()}")
    print(f"체결 발생 횟수: {(trade_vol > 0).sum()}회 / {len(df)}초")